# Akili Drive Evidence Collector v1

Scans `/content/drive/MyDrive/AKM_CLR` without changing source runs. It creates a GitHub-ready evidence ZIP, separates publication candidates from development evidence, scans small files for secrets, excludes weights by default, and writes SHA-256 checksums.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount(os.getenv("AKILI_COLLECTOR_DRIVE_MOUNT", "/content/drive"))
else:
    print("Not running in Colab; Drive mount skipped.")


In [ ]:
import ast, importlib, sys
from pathlib import Path

MODULE_NAME = "akili_drive_evidence_collector_v1"
MODULE_SOURCE = '\nfrom __future__ import annotations\n\nimport csv\nimport dataclasses\nimport datetime as dt\nimport hashlib\nimport json\nimport re\nimport shutil\nimport zipfile\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nPROTOCOL = "akili-drive-evidence-collector-v1"\n\nSAFE_EXTENSIONS = {\n    ".json", ".jsonl", ".csv", ".md", ".txt", ".html", ".htm",\n    ".gif", ".mp4", ".webm", ".png", ".jpg", ".jpeg", ".webp",\n    ".yaml", ".yml", ".toml", ".ipynb", ".log",\n}\nWEIGHT_EXTENSIONS = {".safetensors", ".pt", ".pth", ".bin", ".ckpt", ".onnx"}\nEXCLUDED_DIR_NAMES = {\n    ".git", "__pycache__", ".ipynb_checkpoints", "wandb", "cache",\n    "huggingface", "datasets_cache",\n}\nSECRET_PATTERNS = {\n    "huggingface_token": re.compile(rb"hf_[A-Za-z0-9]{20,}"),\n    "openai_key": re.compile(rb"sk-[A-Za-z0-9_-]{20,}"),\n    "aws_access_key": re.compile(rb"AKIA[0-9A-Z]{16}"),\n    "github_token": re.compile(rb"gh[pousr]_[A-Za-z0-9]{20,}"),\n}\nRUN_HINTS = ("run_", "seed_", "publication", "result", "output")\n\n\ndef utc_now() -> str:\n    return dt.datetime.now(dt.timezone.utc).isoformat()\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef read_json(path: Path) -> Optional[Dict[str, Any]]:\n    try:\n        payload = json.loads(path.read_text(encoding="utf-8"))\n        return payload if isinstance(payload, dict) else None\n    except Exception:\n        return None\n\n\ndef bool_from_hard_checks(path: Path) -> Optional[bool]:\n    payload = read_json(path)\n    if payload is None:\n        return None\n    if isinstance(payload.get("all_passed"), bool):\n        return bool(payload["all_passed"])\n    bools = [value for value in payload.values() if isinstance(value, bool)]\n    return all(bools) if bools else None\n\n\ndef family_from_path(path: Path) -> str:\n    text = path.as_posix().lower()\n    if "agent_evolution" in text:\n        return "agent_evolution"\n    if "robot" in text or "mujoco" in text or "arm_" in text:\n        return "robotics_mujoco"\n    if "skill_runtime" in text or "llm" in text:\n        return "llm_skill_runtime"\n    if "cifar" in text or "phase2" in text or "v0_23" in text:\n        return "cifar100"\n    if "derpp" in text or "mammoth" in text:\n        return "mammoth_baselines"\n    return "other"\n\n\ndef find_run_roots(project_root: Path) -> List[Path]:\n    candidates: set[Path] = set()\n    for path in project_root.rglob("*"):\n        if not path.is_dir():\n            continue\n        if any(part in EXCLUDED_DIR_NAMES for part in path.parts):\n            continue\n        try:\n            child_names = {child.name for child in path.iterdir() if child.is_file()}\n        except OSError:\n            continue\n        has_evidence = bool(\n            child_names\n            & {\n                "summary.json", "hard_checks.json", "registry.json",\n                "audit_chain.jsonl", "audit_log.json",\n                "FINALIZATION_COMPLETE.json",\n                "akili_robotics_v0_2_report.json",\n            }\n        )\n        hinted = any(hint in path.name.lower() for hint in RUN_HINTS)\n        if has_evidence or (hinted and any(path.glob("*.json"))):\n            candidates.add(path.resolve())\n\n    ordered = sorted(candidates, key=lambda p: len(p.parts), reverse=True)\n    selected: List[Path] = []\n    for candidate in ordered:\n        if not any(candidate in existing.parents for existing in selected):\n            selected.append(candidate)\n    return sorted(selected)\n\n\ndef detect_summary(run_root: Path) -> Optional[Path]:\n    preferred = [\n        run_root / "summary.json",\n        run_root / "FINALIZATION_COMPLETE.json",\n        run_root / "akili_robotics_v0_2_report.json",\n    ]\n    for path in preferred:\n        if path.is_file():\n            return path\n    reports = sorted(run_root.glob("*report*.json"))\n    return reports[0] if reports else None\n\n\ndef detect_hard_checks(run_root: Path) -> Optional[Path]:\n    direct = run_root / "hard_checks.json"\n    if direct.is_file():\n        return direct\n    matches = sorted(run_root.rglob("hard_checks.json"))\n    return matches[0] if matches else None\n\n\ndef evidence_flags(run_root: Path) -> Dict[str, bool]:\n    files = [path for path in run_root.rglob("*") if path.is_file()]\n    names = {path.name for path in files}\n    suffixes = {path.suffix.lower() for path in files}\n    return {\n        "has_summary": detect_summary(run_root) is not None,\n        "has_hard_checks": detect_hard_checks(run_root) is not None,\n        "has_registry": "registry.json" in names,\n        "has_audit": bool({"audit_chain.jsonl", "audit_log.json", "audit_chain.json"} & names),\n        "has_config": bool(\n            {"resolved_config.json", "config.json", "environment.json", "environment_versions.json"} & names\n        ),\n        "has_media": bool(suffixes & {".gif", ".mp4", ".webm", ".png"}),\n        "has_csv": ".csv" in suffixes,\n        "has_json_results": any(\n            path.suffix.lower() == ".json"\n            and any(token in path.name.lower() for token in ("result", "report", "summary", "metric"))\n            for path in files\n        ),\n        "has_weights": bool(suffixes & WEIGHT_EXTENSIONS),\n    }\n\n\n@dataclass\nclass RunRecord:\n    run_root: str\n    relative_path: str\n    family: str\n    modified_utc: str\n    total_files: int\n    total_bytes: int\n    all_passed: Optional[bool]\n    status: str\n    flags: Dict[str, bool]\n    summary_path: Optional[str]\n    hard_checks_path: Optional[str]\n    missing_for_publication: List[str]\n\n    def public(self) -> Dict[str, Any]:\n        return dataclasses.asdict(self)\n\n\ndef inspect_run(project_root: Path, run_root: Path) -> RunRecord:\n    files = [path for path in run_root.rglob("*") if path.is_file()]\n    flags = evidence_flags(run_root)\n    hard_path = detect_hard_checks(run_root)\n    summary_path = detect_summary(run_root)\n    all_passed = bool_from_hard_checks(hard_path) if hard_path else None\n\n    missing = []\n    for key, label in (\n        ("has_summary", "summary/report"),\n        ("has_hard_checks", "hard_checks.json"),\n        ("has_config", "resolved configuration/environment"),\n    ):\n        if not flags[key]:\n            missing.append(label)\n    if flags["has_registry"] and not flags["has_audit"]:\n        missing.append("audit chain/log")\n\n    if all_passed is True and not missing:\n        status = "publication_candidate"\n    elif flags["has_summary"] or flags["has_hard_checks"] or flags["has_json_results"]:\n        status = "development_evidence"\n    else:\n        status = "unfinished_or_unknown"\n\n    return RunRecord(\n        run_root=str(run_root),\n        relative_path=run_root.relative_to(project_root).as_posix(),\n        family=family_from_path(run_root),\n        modified_utc=dt.datetime.fromtimestamp(\n            run_root.stat().st_mtime, tz=dt.timezone.utc\n        ).isoformat(),\n        total_files=len(files),\n        total_bytes=sum(path.stat().st_size for path in files),\n        all_passed=all_passed,\n        status=status,\n        flags=flags,\n        summary_path=str(summary_path) if summary_path else None,\n        hard_checks_path=str(hard_path) if hard_path else None,\n        missing_for_publication=missing,\n    )\n\n\ndef scan_secret(path: Path, maximum_scan_bytes: int = 10 * 1024 * 1024) -> List[str]:\n    if path.stat().st_size > maximum_scan_bytes:\n        return []\n    try:\n        data = path.read_bytes()\n    except OSError:\n        return ["unreadable"]\n    return [name for name, pattern in SECRET_PATTERNS.items() if pattern.search(data)]\n\n\ndef should_export(\n    path: Path,\n    *,\n    include_weights: bool,\n    maximum_file_bytes: int,\n) -> Tuple[bool, str]:\n    if any(part in EXCLUDED_DIR_NAMES for part in path.parts):\n        return False, "excluded_directory"\n    if path.stat().st_size > maximum_file_bytes:\n        return False, "file_too_large"\n    suffix = path.suffix.lower()\n    if suffix in WEIGHT_EXTENSIONS and not include_weights:\n        return False, "weight_excluded"\n    name_lower = path.name.lower()\n    if any(token in name_lower for token in ("token", "secret", "credential", "apikey", "api_key")):\n        return False, "sensitive_filename"\n    if suffix not in SAFE_EXTENSIONS and not (include_weights and suffix in WEIGHT_EXTENSIONS):\n        return False, "extension_not_public"\n    hits = scan_secret(path)\n    if hits:\n        return False, "possible_secret:" + ",".join(hits)\n    return True, "included"\n\n\ndef export_runs(\n    project_root: Path,\n    records: Sequence[RunRecord],\n    output_root: Path,\n    *,\n    include_weights: bool,\n    maximum_file_mb: float,\n) -> Dict[str, Any]:\n    staging = output_root / "github_ready"\n    if staging.exists():\n        shutil.rmtree(staging)\n    publication_root = staging / "results" / "publication_import"\n    development_root = staging / "results" / "development_import"\n    publication_root.mkdir(parents=True, exist_ok=True)\n    development_root.mkdir(parents=True, exist_ok=True)\n\n    maximum_file_bytes = int(maximum_file_mb * 1024 * 1024)\n    manifest: List[Dict[str, Any]] = []\n    excluded: List[Dict[str, Any]] = []\n\n    for record in records:\n        source_root = Path(record.run_root)\n        destination_base = (\n            publication_root if record.status == "publication_candidate"\n            else development_root\n        ) / record.family / source_root.name\n        for source in sorted(path for path in source_root.rglob("*") if path.is_file()):\n            allowed, reason = should_export(\n                source,\n                include_weights=include_weights,\n                maximum_file_bytes=maximum_file_bytes,\n            )\n            relative_inside_run = source.relative_to(source_root)\n            if not allowed:\n                excluded.append({\n                    "run": record.relative_path,\n                    "path": relative_inside_run.as_posix(),\n                    "reason": reason,\n                    "bytes": source.stat().st_size,\n                })\n                continue\n            target = destination_base / relative_inside_run\n            target.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(source, target)\n            manifest.append({\n                "run": record.relative_path,\n                "source_relative": source.relative_to(project_root).as_posix(),\n                "export_relative": target.relative_to(staging).as_posix(),\n                "bytes": source.stat().st_size,\n                "sha256": sha256_file(source),\n                "status": record.status,\n            })\n\n        status_path = destination_base / "PUBLICATION_STATUS.json"\n        status_path.parent.mkdir(parents=True, exist_ok=True)\n        status_path.write_text(\n            json.dumps(record.public(), indent=2, sort_keys=True),\n            encoding="utf-8",\n        )\n\n    inventory = {\n        "protocol": PROTOCOL,\n        "created_at": utc_now(),\n        "project_root": str(project_root),\n        "include_weights": include_weights,\n        "maximum_file_mb": maximum_file_mb,\n        "runs": [record.public() for record in records],\n        "included_files": manifest,\n        "excluded_files": excluded,\n    }\n    (staging / "DRIVE_INVENTORY.json").write_text(\n        json.dumps(inventory, indent=2, sort_keys=True), encoding="utf-8"\n    )\n\n    with (staging / "DRIVE_INVENTORY.csv").open("w", encoding="utf-8", newline="") as handle:\n        fieldnames = [\n            "relative_path", "family", "modified_utc", "total_files", "total_bytes",\n            "all_passed", "status", "missing_for_publication",\n        ]\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\n        writer.writeheader()\n        for record in records:\n            writer.writerow({\n                "relative_path": record.relative_path,\n                "family": record.family,\n                "modified_utc": record.modified_utc,\n                "total_files": record.total_files,\n                "total_bytes": record.total_bytes,\n                "all_passed": record.all_passed,\n                "status": record.status,\n                "missing_for_publication": "; ".join(record.missing_for_publication),\n            })\n\n    lines = [\n        "# Akili Drive evidence inventory",\n        "",\n        f"Generated: {inventory[\'created_at\']}",\n        "",\n        "| Status | Family | Run | all_passed | Missing |",\n        "|---|---|---|---:|---|",\n    ]\n    for record in records:\n        lines.append(\n            f"| {record.status} | {record.family} | `{record.relative_path}` | "\n            f"{record.all_passed} | {\', \'.join(record.missing_for_publication) or \'—\'} |"\n        )\n    (staging / "PUBLICATION_CANDIDATES.md").write_text(\n        "\\n".join(lines) + "\\n", encoding="utf-8"\n    )\n\n    checksums = []\n    for path in sorted(p for p in staging.rglob("*") if p.is_file()):\n        checksums.append(f"{sha256_file(path)}  {path.relative_to(staging).as_posix()}")\n    (staging / "SHA256SUMS").write_text("\\n".join(checksums) + "\\n", encoding="utf-8")\n\n    output_root.mkdir(parents=True, exist_ok=True)\n    zip_path = output_root / (\n        "akili_github_evidence_"\n        + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n        + ".zip"\n    )\n    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(p for p in staging.rglob("*") if p.is_file()):\n            archive.write(path, path.relative_to(staging).as_posix())\n\n    return {\n        "protocol": PROTOCOL,\n        "project_root": str(project_root),\n        "output_root": str(output_root),\n        "zip_path": str(zip_path),\n        "staging_root": str(staging),\n        "runs_found": len(records),\n        "publication_candidates": sum(\n            record.status == "publication_candidate" for record in records\n        ),\n        "development_evidence": sum(\n            record.status == "development_evidence" for record in records\n        ),\n        "included_files": len(manifest),\n        "excluded_files": len(excluded),\n    }\n\n\ndef collect(\n    project_root: Path,\n    output_root: Path,\n    *,\n    include_weights: bool = False,\n    maximum_file_mb: float = 75.0,\n) -> Dict[str, Any]:\n    project_root = project_root.expanduser().resolve()\n    if not project_root.is_dir():\n        raise FileNotFoundError(project_root)\n    run_roots = find_run_roots(project_root)\n    records = [inspect_run(project_root, run_root) for run_root in run_roots]\n    return export_runs(\n        project_root,\n        records,\n        output_root.expanduser().resolve(),\n        include_weights=include_weights,\n        maximum_file_mb=maximum_file_mb,\n    )\n\n\ndef synthetic_verification(root: Path) -> Dict[str, Any]:\n    if root.exists():\n        shutil.rmtree(root)\n    project = root / "AKM_CLR"\n    good = project / "stage05" / "akili_robotics_v0_2_mujoco" / "run_good"\n    partial = project / "stage05" / "akili_skill_runtime_v0_1" / "run_partial"\n    good.mkdir(parents=True)\n    partial.mkdir(parents=True)\n\n    (good / "summary.json").write_text(\'{"score": 1.0}\', encoding="utf-8")\n    (good / "hard_checks.json").write_text(\'{"all_passed": true}\', encoding="utf-8")\n    (good / "resolved_config.json").write_text(\'{"seed": 1}\', encoding="utf-8")\n    (good / "registry.json").write_text(\'{"skills": {}}\', encoding="utf-8")\n    (good / "audit_log.json").write_text(\'[]\', encoding="utf-8")\n    (good / "demo.gif").write_bytes(b"GIF89a")\n    (good / "adapter_model.safetensors").write_bytes(b"weight")\n\n    (partial / "summary.json").write_text(\'{"score": 0.5}\', encoding="utf-8")\n    (partial / "hard_checks.json").write_text(\'{"all_passed": false}\', encoding="utf-8")\n    (partial / "resolved_config.json").write_text(\'{"seed": 1}\', encoding="utf-8")\n\n    result = collect(project, root / "exports", include_weights=False, maximum_file_mb=5)\n    with zipfile.ZipFile(result["zip_path"]) as archive:\n        names = set(archive.namelist())\n    checks = {\n        "two_runs_found": result["runs_found"] == 2,\n        "one_publication_candidate": result["publication_candidates"] == 1,\n        "one_development_run": result["development_evidence"] == 1,\n        "weights_excluded": not any(name.endswith(".safetensors") for name in names),\n        "gif_included": any(name.endswith("demo.gif") for name in names),\n        "inventory_written": "DRIVE_INVENTORY.json" in names,\n        "checksums_written": "SHA256SUMS" in names,\n    }\n    return {"passed": all(checks.values()), "checks": checks, "result": result}\n'
runtime_root = Path("/tmp/akili_evidence_collector")
runtime_root.mkdir(parents=True, exist_ok=True)
module_path = runtime_root / f"{MODULE_NAME}.py"
module_path.write_text(MODULE_SOURCE, encoding="utf-8")
ast.parse(MODULE_SOURCE)
compile(MODULE_SOURCE, str(module_path), "exec")
if str(runtime_root) not in sys.path:
    sys.path.insert(0, str(runtime_root))
collector = importlib.import_module(MODULE_NAME)
print("Protocol:", collector.PROTOCOL)


In [ ]:
import json, shutil
from pathlib import Path
test_root = Path("/tmp/akili_collector_verification")
shutil.rmtree(test_root, ignore_errors=True)
verification = collector.synthetic_verification(test_root)
assert verification["passed"], verification
print(json.dumps(verification["checks"], indent=2))


In [ ]:
import os
from pathlib import Path
project_root = Path(os.getenv(
    "AKILI_COLLECTOR_PROJECT_ROOT",
    "/content/drive/MyDrive/AKM_CLR",
))
output_root = Path(os.getenv(
    "AKILI_COLLECTOR_OUTPUT_ROOT",
    "/content/drive/MyDrive/AKM_CLR/publication_exports",
))
include_weights = os.getenv(
    "AKILI_COLLECTOR_INCLUDE_WEIGHTS", "0"
).strip().lower() in {"1", "true", "yes"}
maximum_file_mb = float(os.getenv("AKILI_COLLECTOR_MAX_FILE_MB", "75"))
print({
    "project_root": str(project_root),
    "output_root": str(output_root),
    "include_weights": include_weights,
    "maximum_file_mb": maximum_file_mb,
})


In [ ]:
import json, os
SKIP_REAL = os.getenv("AKILI_COLLECTOR_SKIP_REAL", "0").strip().lower() in {
    "1", "true", "yes"
}
if SKIP_REAL:
    RESULT = None
    print("Real Drive scan skipped.")
else:
    RESULT = collector.collect(
        project_root,
        output_root,
        include_weights=include_weights,
        maximum_file_mb=maximum_file_mb,
    )
    print(json.dumps(RESULT, indent=2))


In [ ]:
if RESULT is not None:
    print("Archive:", RESULT["zip_path"])
    if "google.colab" in sys.modules and os.getenv(
        "AKILI_COLLECTOR_AUTO_DOWNLOAD", "1"
    ).strip().lower() in {"1", "true", "yes"}:
        from google.colab import files
        files.download(RESULT["zip_path"])
